In [ ]:
# -*- coding: utf-8 -*-
"""
Sticker cleaner + QA para 'HOJA DE VIDA' con código de barras.
- Lee todos los JSON dentro de JSON_FOLDER (formato [ {pagina, imagen, texto}, ... ] o {"paginas": [...] }).
- NO re-OCRiza. Limpia líneas del sticker y baja su sesgo.
- Clasifica páginas en categorías QA y genera reportes y ejemplos para auditoría.

Salidas por archivo:
  <nombre>_clean.json
  <nombre>_clean_report.csv
  <nombre>_debug_sticker.csv

Salidas globales en OUTPUT_DIR:
  sticker_QA_examples.csv         (todos los ejemplos)
  sticker_QA_examples.jsonl       (uno por línea)
  sticker_QA_summary.csv          (conteo por categoría y por archivo)
  (opcional) /examples/<CATEGORIA>/*.png  (copia de imágenes)

Ajusta JSON_FOLDER, OUTPUT_DIR y COPY_IMAGES.
"""

from __future__ import annotations
from pathlib import Path
import json, re, csv, shutil
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

# ============================================================
# RUTAS / FLAGS (ajusta AQUÍ)
# ============================================================
JSON_FOLDER = r"D:\historias\dev\ocr_por_doc\LETRA A\ACTA N° 70\1" 
OUTPUT_DIR  = r"C:\Users\juans\Documents\version_final_historias laborales\answert"
COPY_IMAGES = False                                  # True = copia imágenes por categoría

DO_RUN = True
DEBUG = True
FORCE_SAVE_REPORTS_EVEN_IF_EMPTY = True

# ============================================================
# CONFIGURACIÓN DEL DETECTOR
# ============================================================

@dataclass
class StickerConfig:
    # ventanas y límites (más estrictos)
    neighbor_radius: int = 1                 # solo vecinas inmediatas
    min_digits_for_barcode: int = 12
    min_digit_ratio_line: float = 0.85
    max_alpha_ratio_line: float = 0.20
    max_tokens_for_sticker_line: int = 3
    sticker_line_score_thresh: float = 0.70

    # distancia máxima entre 'HV' y número para formar bloque
    max_gap_lines: int = 6
    # restringe a zonas superior/inferior (sticker suele estar en bordes)
    zone_fraction: float = 0.30              # 30% superior o inferior
    # válvulas de seguridad
    block_size_limit: int = 5                # nunca quitar más de 5 líneas
    max_removed_ratio: float = 0.15          # ni >15% de líneas de la página

    # 'Hoja de Vida'
    hv_regex: re.Pattern = re.compile(r"\bhoja\s*de\s*vida\b", re.I | re.U)

    hv_support_terms: Tuple[re.Pattern, ...] = (
        re.compile(r"\bfunci[oó]n\s+p[úu]blica\b", re.I),
        re.compile(r"\bsigep\b", re.I),
        re.compile(r"\bformato\s+([úu]nico\s+de\s+)?hoja\s*de\s*vida\b", re.I),
        re.compile(r"\bdatos\s+personales\b", re.I),
    )

    long_number_patterns: Tuple[re.Pattern, ...] = (
        re.compile(r"(?<!\d)(\d{11,})(?!\d)"),
        re.compile(r"\b\d{2,}(?:[^\w\s]\d{2,}){2,}\d{2,}\b"),
        re.compile(r"(?:\d[\W_]?){12,}")
    )

    # detección fuerte de HV Función Pública
    hv_fp_regexes: Tuple[re.Pattern, ...] = (
        re.compile(r"\bformato\s+[úu]nico\s+de\s+hoja\s*de\s*vida\b", re.I),
        re.compile(r"\bhoja\s*de\s*vida.*funci[oó]n\s+p[úu]blica\b", re.I),
    )

    # desactiva el pareo laxo por defecto (lo reimplementamos controlado)
    enable_loose_pairing: bool = False



CFG = StickerConfig()

# ============================================================
# UTILIDADES DE TEXTO
# ============================================================

def normalize_ocr_text(t: Optional[str]) -> str:
    if not t:
        return ""
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[·•◦▪●■□▫▶►♦●]", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = "\n".join(ln.strip() for ln in t.split("\n"))
    return t



def is_sticker_like_number(line: str, cfg: StickerConfig = CFG) -> bool:
    """Número de sticker: patrón largo ó muchos dígitos en línea corta y poco alfabeto."""
    d_ratio, a_ratio, _ = char_ratio(line)
    if has_long_number_patterns(line, cfg):
        return True
    if digits_only_len(line) >= cfg.min_digits_for_barcode and a_ratio <= cfg.max_alpha_ratio_line and token_count(line) <= cfg.max_tokens_for_sticker_line:
        return True
    return False

_CONTENT_WORDS = re.compile(
    r"\b(nombrad[oa]|cargo|oficio|resolu(?:ci[oó]n)?|fecha|ciudad|direcci[oó]n|tel[eé]fono|departamento|"
    r"certificad[oa]|aprobaci[oó]n|observaci[oó]n|concepto|historia|ingreso|padre|madre|c[oó]nyuge|"
    r"barrio|correo|profesi[oó]n|estudios|empresa)\b",
    re.I
)

def is_content_rich_line(line: str) -> bool:
    """Línea de texto real: muchas letras/palabras o vocabulario administrativo típico."""
    d_ratio, a_ratio, _ = char_ratio(line)
    return (token_count(line) >= 6 and a_ratio >= 0.55) or bool(_CONTENT_WORDS.search(line))

def in_top_or_bottom_zone(idx: int, n_lines: int, frac: float) -> bool:
    zone = max(1, int(frac * n_lines))
    return idx < zone or idx >= (n_lines - zone)

def split_lines(t: str) -> List[str]:
    return t.split("\n") if t else []

def digits_only_len(s: str) -> int:
    return sum(ch.isdigit() for ch in s)

def token_count(s: str) -> int:
    return len([x for x in s.strip().split(" ") if x])

def char_ratio(s: str) -> Tuple[float, float, float]:
    n = max(len(s), 1)
    return (
        sum(c.isdigit() for c in s) / n,
        sum(c.isalpha() for c in s) / n,
        s.count("-") / n
    )

def collapse_letters(s: str) -> str:
    return re.sub(r"[^A-ZÁÉÍÓÚÜÑ]", "", s.upper())

def is_hv_robust(line: str, cfg: StickerConfig = CFG) -> bool:
    return bool(cfg.hv_regex.search(line) or ("HOJADEVIDA" in collapse_letters(line)))

def has_long_number_patterns(s: str, cfg: StickerConfig = CFG) -> bool:
    return any(p.search(s) for p in cfg.long_number_patterns)

def window_digits_count(lines: List[str], center: int, radius: int) -> int:
    i0, i1 = max(0, center - radius), min(len(lines), center + radius + 1)
    return digits_only_len("".join(lines[i0:i1]))

# ============================================================
# DETECCIÓN DE STICKER
# ============================================================

def score_line_as_sticker(line: str, cfg: StickerConfig = CFG) -> float:
    has_hdv = 1.0 if is_hv_robust(line, cfg) else 0.0
    d_ratio, a_ratio, h_ratio = char_ratio(line)
    tok = token_count(line)
    long_num = 1.0 if has_long_number_patterns(line, cfg) else 0.0
    score = (
        0.40 * has_hdv +
        0.25 * (1.0 if d_ratio >= cfg.min_digit_ratio_line else 0.0) +
        0.10 * (1.0 if h_ratio >= 0.08 else 0.0) +
        0.20 * long_num +
        0.05 * (1.0 if tok <= cfg.max_tokens_for_sticker_line else 0.0)
    )
    return score

def find_sticker_block(lines: List[str], cfg: StickerConfig = CFG) -> Tuple[bool, List[int], bool]:
    """
    Solo elimina las líneas realmente del sticker:
      - 1 línea 'HOJA DE VIDA' (o variante robusta) y
      - 1 línea con número de sticker (is_sticker_like_number),
    ± vecinas inmediatas que también parezcan sticker (NO texto rico).
    Requiere que ambas estén cerca (<= max_gap_lines) o en zona superior/inferior.
    Aplica límites de tamaño y proporción removida.
    """
    n = len(lines)
    if n == 0:
        return False, [], False

    # índices candidatos
    hv_idxs = [i for i, ln in enumerate(lines) if is_hv_robust(ln, cfg)]
    num_idxs = [i for i, ln in enumerate(lines) if is_sticker_like_number(ln, cfg)]

    if not hv_idxs or not num_idxs:
        return False, [], False

    # empareja por distancia mínima
    best_pair = None
    best_dist = 10**9
    for h in hv_idxs:
        for m in num_idxs:
            dist = abs(h - m)
            if dist < best_dist:
                best_dist = dist
                best_pair = (h, m)

    if best_pair is None:
        return False, [], False

    h, m = best_pair
    # debe estar cerca o en zonas borde
    close_enough = best_dist <= cfg.max_gap_lines
    edge_zones = in_top_or_bottom_zone(h, n, cfg.zone_fraction) and in_top_or_bottom_zone(m, n, cfg.zone_fraction)
    if not (close_enough or edge_zones):
        return False, [], False

    # bloque inicial: solo HV y número
    block = {h, m}

    # expande ±1 SOLO si la línea vecina también parece sticker y no es "content rich"
    for idx in list(block):
        for j in (idx - cfg.neighbor_radius, idx + cfg.neighbor_radius):
            if 0 <= j < n:
                ln = lines[j]
                if not is_content_rich_line(ln) and (is_hv_robust(ln, cfg) or is_sticker_like_number(ln, cfg)):
                    block.add(j)

    # opcional: si HV y número están muy cerca, inspecciona líneas intermedias (<=3) y añade las que sean "sticker-like"
    if best_dist <= 3:
        lo, hi = sorted((h, m))
        for j in range(lo + 1, hi):
            if not is_content_rich_line(lines[j]) and (is_hv_robust(lines[j], cfg) or is_sticker_like_number(lines[j], cfg)):
                block.add(j)

    # límites de seguridad
    if len(block) > cfg.block_size_limit or (len(block) / max(1, n)) > cfg.max_removed_ratio:
        # demasiado agresivo: conserva SOLO hv y número
        block = {h, m}

    hv_in_sticker = h in block
    removed_idxs = sorted(block)
    return True, removed_idxs, hv_in_sticker


# ============================================================
# LIMPIEZA + BLINDAJE
# ============================================================

def remove_lines_by_idx(lines: List[str], idxs: List[int]) -> List[str]:
    idxs_set = set(idxs)
    return [ln for i, ln in enumerate(lines) if i not in idxs_set]

def hv_outside_sticker(lines: List[str], idxs_removed: List[int], cfg: StickerConfig = CFG) -> bool:
    removed = set(idxs_removed)
    for i, ln in enumerate(lines):
        if i in removed:
            continue
        if is_hv_robust(ln, cfg):
            return True
    return False

def hv_safe_text(lines: List[str], idxs_removed: List[int], cfg: StickerConfig = CFG) -> bool:
    """Considera HV solo si hay 2+ evidencias textuales fuera del bloque."""
    if not hv_outside_sticker(lines, idxs_removed, cfg):
        return False
    clean = "\n".join(remove_lines_by_idx(lines, idxs_removed))
    hits = sum(1 for p in cfg.hv_support_terms if p.search(clean))
    return hits >= 2

def is_hv_fp_confident(clean_text: str, cfg: StickerConfig = CFG) -> bool:
    """Alta confianza para 'Hoja de Vida – Función Pública'."""
    return any(p.search(clean_text) for p in cfg.hv_fp_regexes)

def extract_removed_text(lines: List[str], idxs_removed: List[int], window:int=0) -> str:
    """Devuelve texto de líneas removidas (y opcionalmente contexto)."""
    n = len(lines)
    parts = []
    for i in idxs_removed:
        i0 = max(0, i - window); i1 = min(n, i + window + 1)
        segment = " | ".join(lines[i0:i1])
        parts.append(f"[{i}] {segment}")
    return " || ".join(parts)

def make_clean_text(ocr_text_raw: str, cfg: StickerConfig = CFG) -> Dict:
    norm = normalize_ocr_text(ocr_text_raw)
    lines = split_lines(norm)
    barcode_flag, rm_idxs, hv_in = find_sticker_block(lines, cfg=cfg)
    lines_clean = remove_lines_by_idx(lines, rm_idxs)
    clean_text = "\n".join(lines_clean)
    hv_out = hv_outside_sticker(lines, rm_idxs, cfg=cfg)
    hv_safe = hv_safe_text(lines, rm_idxs, cfg=cfg)
    hv_fp = is_hv_fp_confident(clean_text, cfg=cfg)
    removed_text = extract_removed_text(lines, rm_idxs, window=1)
    return {
        "barcode_flag": barcode_flag,
        "removed_line_idxs": rm_idxs,
        "removed_text_preview": removed_text,
        "hv_in_sticker": hv_in,
        "hv_outside_sticker": hv_out,
        "hv_safe_text": hv_safe,
        "hv_funcion_publica_confident": hv_fp,
        "ocr_text_raw": ocr_text_raw,
        "ocr_text_clean": clean_text
    }

# ============================================================
# CATEGORIZACIÓN QA
# ============================================================

QA_CATEGORIES = (
    "STICKER_ONLY",            # sticker detectado y HV solo dentro del bloque
    "BARCODE_ONLY",            # número largo sin HV
    "STICKER_AND_HV_UNCERTAIN",# HV fuera del bloque pero sin soportes suficientes
    "HV_REAL_TEXTUAL",         # HV fuera del bloque + soportes suficientes (probable real)
    "HV_FUNCION_PUBLICA",      # caso fuerte de HV Función Pública
    "LONG_NUMBER_ONLY",        # no sticker, pero hay números largos dispersos
    "NO_STICKER"               # nada relevante
)

def has_any_long_number(text: str, cfg: StickerConfig = CFG) -> bool:
    return any(p.search(text) for p in cfg.long_number_patterns)

def categorize_page(res: Dict, cfg: StickerConfig = CFG) -> str:
    b = res["barcode_flag"]
    hv_in = res["hv_in_sticker"]
    hv_out = res["hv_outside_sticker"]
    hv_safe = res["hv_safe_text"]
    hv_fp = res["hv_funcion_publica_confident"]
    raw = res["ocr_text_raw"] or ""
    clean = res["ocr_text_clean"] or ""

    if b:
        if hv_in and not hv_out:
            return "STICKER_ONLY"
        if hv_in and hv_out and not hv_safe:
            return "STICKER_AND_HV_UNCERTAIN"
        if hv_safe and hv_fp:
            return "HV_FUNCION_PUBLICA"
        if hv_safe:
            return "HV_REAL_TEXTUAL"
        # número sin HV
        return "BARCODE_ONLY"
    else:
        if hv_fp and hv_safe:
            return "HV_FUNCION_PUBLICA"
        if hv_safe:
            return "HV_REAL_TEXTUAL"
        if has_any_long_number(raw, cfg) or has_any_long_number(clean, cfg):
            return "LONG_NUMBER_ONLY"
        return "NO_STICKER"

# ============================================================
# I/O Y PROCESO
# ============================================================

def safe_read_json(fp: Path):
    txt = fp.read_text(encoding="utf-8", errors="replace")
    data = json.loads(txt)
    if isinstance(data, dict) and "paginas" in data:
        data = data["paginas"]
    if not isinstance(data, list):
        raise ValueError("El JSON debe ser una lista de páginas o un dict con clave 'paginas'.")
    return data

def load_histories(folder: Path) -> List[Tuple[Path, List[Dict]]]:
    out = []
    for fp in sorted(folder.glob("*.json")):
        if fp.name.endswith("_clean.json") or fp.name.endswith("_clean_report.json"):
            continue
        try:
            data = safe_read_json(fp)
            out.append((fp, data))
        except Exception as e:
            print(f"[WARN] No pude leer {fp.name}: {e}")
    return out

def debug_dump_lines(file_name: str, lines: List[str], rm_idxs: List[int], folder: Path):
    out_csv = folder / f"{Path(file_name).stem}_debug_sticker.csv"
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["idx","removed","digit_ratio","alpha_ratio","hyphen_ratio","long_number","has_HV","line"])
        rm = set(rm_idxs)
        for i, ln in enumerate(lines):
            d,a,h = char_ratio(ln)
            w.writerow([
                i, int(i in rm),
                round(d,3), round(a,3), round(h,3),
                int(has_long_number_patterns(ln)), int(is_hv_robust(ln)), ln
            ])

def process_history_json(pages: List[Dict], cfg: StickerConfig = CFG, dbg_path: Optional[Path]=None) -> List[Dict]:
    results = []
    for row in pages:
        texto = row.get("texto", "") or ""
        norm = normalize_ocr_text(texto)
        lines = split_lines(norm)

        # detectar + limpiar
        barcode_flag, rm_idxs, hv_in = find_sticker_block(lines, cfg=cfg)
        lines_clean = remove_lines_by_idx(lines, rm_idxs)
        clean_text = "\n".join(lines_clean)
        hv_out = hv_outside_sticker(lines, rm_idxs, cfg=cfg)
        hv_safe = hv_safe_text(lines, rm_idxs, cfg=cfg)
        hv_fp = is_hv_fp_confident(clean_text, cfg=cfg)
        removed_text = extract_removed_text(lines, rm_idxs, window=1)

        if DEBUG and dbg_path is not None:
            try:
                debug_dump_lines(dbg_path.name, lines, rm_idxs, dbg_path.parent)
            except Exception as e:
                print(f"[WARN] No pude escribir debug para {dbg_path.name}: {e}")

        res = {
            "pagina": row.get("pagina"),
            "imagen": row.get("imagen"),
            "barcode_flag": barcode_flag,
            "removed_line_idxs": rm_idxs,
            "removed_text_preview": removed_text,
            "hv_in_sticker": hv_in,
            "hv_outside_sticker": hv_out,
            "hv_safe_text": hv_safe,
            "hv_funcion_publica_confident": hv_fp,
            "ocr_text_raw": texto,
            "ocr_text_clean": clean_text
        }
        res["qa_category"] = categorize_page(res, cfg=cfg)
        results.append(res)
    return results

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def copy_image_safe(src: Optional[str], dst_dir: Path):
    if not src:
        return
    try:
        src_p = Path(src)
        if src_p.exists():
            ensure_dir(dst_dir)
            shutil.copy2(src_p, dst_dir / src_p.name)
    except Exception as e:
        print(f"[WARN] No se pudo copiar imagen {src}: {e}")

def process_folder(folder_path: str, output_dir: str, save_outputs: bool = True, copy_images: bool = False) -> List[Dict]:
    folder = Path(folder_path)
    outdir = Path(output_dir)
    ensure_dir(outdir)

    histories = load_histories(folder)
    if not histories:
        print("[INFO] No se encontraron JSON (o todos son salidas _clean).")
        return []

    all_rows: List[Dict] = []
    per_file_counts = defaultdict(Counter)

    for fp, data in histories:
        processed = process_history_json(data, cfg=CFG, dbg_path=fp)

        # guarda por archivo
        if save_outputs or FORCE_SAVE_REPORTS_EVEN_IF_EMPTY:
            out_json = fp.with_name(fp.stem + "_clean.json")
            out_json.write_text(json.dumps(processed, ensure_ascii=False, indent=2), encoding="utf-8")

            out_csv = fp.with_name(fp.stem + "_clean_report.csv")
            with out_csv.open("w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["file","pagina","qa_category","barcode_flag","hv_in_sticker","hv_outside_sticker",
                            "hv_safe_text","hv_funcion_publica_confident","removed_line_idxs"])
                for r in processed:
                    w.writerow([fp.name, r["pagina"], r["qa_category"], r["barcode_flag"], r["hv_in_sticker"],
                                r["hv_outside_sticker"], r["hv_safe_text"], r["hv_funcion_publica_confident"],
                                "|".join(map(str, r["removed_line_idxs"]))])

        # acumula para QA global
        for r in processed:
            rr = {**r, "file": fp.name}
            all_rows.append(rr)
            per_file_counts[fp.name][r["qa_category"]] += 1

            # copia imagen por categoría (opcional)
            if copy_images and r.get("imagen"):
                cat_dir = outdir / "examples" / r["qa_category"]
                copy_image_safe(r["imagen"], cat_dir)

    # --- ejemplos globales ---
    examples_csv = Path(output_dir) / "sticker_QA_examples.csv"
    with examples_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["file","pagina","qa_category","barcode_flag","hv_in_sticker","hv_outside_sticker",
                    "hv_safe_text","hv_funcion_publica_confident","image","removed_text_preview"])
        for r in all_rows:
            w.writerow([r["file"], r["pagina"], r["qa_category"], r["barcode_flag"], r["hv_in_sticker"],
                        r["hv_outside_sticker"], r["hv_safe_text"], r["hv_funcion_publica_confident"],
                        r.get("imagen",""), r.get("removed_text_preview","")])

    examples_jsonl = Path(output_dir) / "sticker_QA_examples.jsonl"
    with examples_jsonl.open("w", encoding="utf-8") as f:
        for r in all_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # --- resumen por archivo ---
    summary_csv = Path(output_dir) / "sticker_QA_summary.csv"
    with summary_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        header = ["file"] + list(QA_CATEGORIES)
        w.writerow(header)
        for fname, cnt in per_file_counts.items():
            row = [fname] + [cnt.get(cat, 0) for cat in QA_CATEGORIES]
            w.writerow(row)

    # impresión corta
    total = len(all_rows)
    flagged = sum(1 for r in all_rows if r["barcode_flag"])
    hvfp = sum(1 for r in all_rows if r["qa_category"] == "HV_FUNCION_PUBLICA")
    print(f"[RESUMEN] archivos: {len(histories)} | páginas: {total} | con_sticker: {flagged} | HV_FP: {hvfp}")

    return all_rows

# ============================================================
# EJECUCIÓN
# ============================================================

if DO_RUN:
    _ = process_folder(JSON_FOLDER, OUTPUT_DIR, save_outputs=True, copy_images=COPY_IMAGES)

[RESUMEN] archivos: 12 | páginas: 500 | con_sticker: 259 | HV_FP: 2
